## Part 1: Restructuring Data

In [27]:
import pandas as pd
non_messy_data_df = pd.read_csv("../data/pima-indians-diabetes.csv")
non_messy_data_df.head()

,times_pregnant,plasma_glucose_concentration,diastolic_blood_pressure,triceps_thickness,2-hour_serum_insulin,BMI,diabetes_pedigreen,age,diabetes
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [28]:
# You should read in this data and restructure it to make it identical to the
# pima-indians-diabetes.csv introduced in the previous topic.
messy_data_df = pd.read_csv(f"../data/messy-pima-indians-diabetes.csv")
messy_data_df.head()

,Non-diabetic
0,times_pregnant6.0000
1,plasma_glucose_concentration148.0000
2,diastolic_blood_pressure72.0000
3,triceps_thickness35.0000
4,2_hour_serum_insulin0.0000


### Get familiar with the messy dataset

In [29]:
def find_unique_columns(df):
    # Drop age and BMI because they are too small and would generate too many combinations
    age = "age"
    BMI = "BMI"
    df_without_age_and_BMI = df[~df["Non-diabetic"].str.startswith((age, BMI))].copy()
    
    # Drop matches up to 12 chars
    df_without_age_and_BMI["prefix"] = df_without_age_and_BMI["Non-diabetic"].str[:12]
    df_without_age_and_BMI = df_without_age_and_BMI.drop_duplicates(subset="prefix")

    # Add back age and BMI
    return pd.concat([df_without_age_and_BMI['Non-diabetic'], pd.Series([age, BMI])]).values

# list columns not existing in original
def non_existing_columns(cols):
    return [ col for col in cols
             if not any(col.startswith(col_orig) for col_orig in non_messy_data_df.columns.to_list())]

def missing_columns(cols):
    missing = [
        col_orig for col_orig in non_messy_data_df.columns.to_list()
        if not any(col_orig in colm for colm in cols)]
    return missing

In [30]:
cols_in_messy = find_unique_columns(messy_data_df)
print("messy:")
print("\n".join(cols_in_messy))
print("-------")
print("original data:")
print("\n".join(non_messy_data_df.columns.to_list()))
print("-------")

messy:
times_pregnant6.0000
plasma_glucose_concentration148.0000
diastolic_blood_pressure72.0000
triceps_thickness35.0000
2_hour_serum_insulin0.0000
diabetes_pedigreen0.6270
diabetes1.0000
Participants10-20
Diabetic
diabetes0.0000
age
BMI
-------
original data:
times_pregnant
plasma_glucose_concentration
diastolic_blood_pressure
triceps_thickness
2-hour_serum_insulin
BMI
diabetes_pedigreen
age
diabetes
-------


I see the following differences:
- 2_hour_serum_insulin instead of 2-hour-serum-insulin
- Participants is an extra column
- Diabetic is an extra column

Let's validate them with code too!

In [31]:
print(non_existing_columns(cols_in_messy))
print(missing_columns(cols_in_messy))

['2_hour_serum_insulin0.0000', 'Participants10-20', 'Diabetic']
['2-hour_serum_insulin']


In [32]:
# Inspect what are those two extra columns
print(messy_data_df [ messy_data_df['Non-diabetic'].str.startswith('Diabetic') ])

# See what's going on around Diabetic:
diabetic_index = messy_data_df[ messy_data_df['Non-diabetic'] == 'Diabetic' ].index[0]
print(messy_data_df.iloc[diabetic_index-3:diabetic_index+3])

# Participants
messy_data_df [ messy_data_df['Non-diabetic'].str.startswith('Participants') ].head()

     Non-diabetic
2756     Diabetic
                             Non-diabetic
2753                           age47.0000
2754                       diabetes1.0000
2755                 times_pregnant1.0000
2756                             Diabetic
2757                 times_pregnant1.0000
2758  plasma_glucose_concentration85.0000


,Non-diabetic
50,Participants10-20
121,Participants20-30
182,Participants30-40
213,Participants40-50
254,Participants50-60


In [33]:
# Split the dataframe around Diabetics, as every second row in the original dataset is added after it
messy_odd_rows = messy_data_df.iloc[:diabetic_index]
messy_even_rows = messy_data_df.iloc[diabetic_index+1:]
messy_even_rows.head()

,Non-diabetic
2757,times_pregnant1.0000
2758,plasma_glucose_concentration85.0000
2759,diastolic_blood_pressure66.0000
2760,triceps_thickness29.0000
2761,2_hour_serum_insulin0.0000


### Check if both dataframes are the same as messy

In [34]:
cols_in_odd = find_unique_columns(messy_odd_rows)
print("Odd:")
print(non_existing_columns(cols_in_odd))
print(missing_columns(cols_in_odd))
print(len(messy_odd_rows))
print("----")

cols_in_even = find_unique_columns(messy_even_rows)
print("Even:")
print(non_existing_columns(cols_in_even))
print(missing_columns(cols_in_even))
print(len(messy_even_rows))

messy_odd_rows = messy_odd_rows.reset_index(drop=True)
messy_even_rows = messy_even_rows.reset_index(drop=True)

Odd:
['2_hour_serum_insulin0.0000', 'Participants10-20']
['2-hour_serum_insulin']
2756
----
Even:
['2_hour_serum_insulin0.0000', 'Participants10-20']
['2-hour_serum_insulin']
5076


Only the Diabetic is missing, so it looks promising

In [35]:
# Rename to the one we have in messy for comparison
changed_col_names = [s.replace('2-hour_serum_insulin', '2_hour_serum_insulin') for s in non_messy_data_df.columns.to_list()]

In [36]:
def extract_col_and_val(row):
    for col in changed_col_names:
        if row.startswith(col):
            # get value after col name
            value = row[len(col):]
            return col, value
    # missing columns
    return None, None

def split_to_col_val(df):
    new_cols = pd.DataFrame()
    new_cols[ ['Column', 'Value'] ] = df['Non-diabetic'].apply(lambda x: pd.Series(extract_col_and_val(x)))
    
    # Drop the non-existing columns
    new_cols = new_cols.dropna(subset = ['Column'])
    new_cols = new_cols.reset_index(drop=True)
    return new_cols

splitted_odds = split_to_col_val(messy_odd_rows)
splitted_evens = split_to_col_val(messy_even_rows)

In [37]:
print(splitted_odds.head(11))
print(splitted_evens.head(11))

                          Column     Value
0                 times_pregnant    6.0000
1   plasma_glucose_concentration  148.0000
2       diastolic_blood_pressure   72.0000
3              triceps_thickness   35.0000
4           2_hour_serum_insulin    0.0000
5                            BMI   33.6000
6             diabetes_pedigreen    0.6270
7                            age   50.0000
8                       diabetes    1.0000
9                 times_pregnant    6.0000
10                times_pregnant    8.0000
                          Column    Value
0                 times_pregnant   1.0000
1   plasma_glucose_concentration  85.0000
2       diastolic_blood_pressure  66.0000
3              triceps_thickness  29.0000
4           2_hour_serum_insulin   0.0000
5                            BMI  26.6000
6             diabetes_pedigreen   0.3510
7                            age  31.0000
8                       diabetes   0.0000
9                 times_pregnant   1.0000
10                time

In [38]:
# Times pregnant it duplicated in both, let's see how
print(splitted_odds[ splitted_odds['Column'] == 'times_pregnant'].head())
print(splitted_evens[ splitted_evens['Column'] == 'times_pregnant'].head())

            Column   Value
0   times_pregnant  6.0000
9   times_pregnant  6.0000
10  times_pregnant  8.0000
19  times_pregnant  8.0000
20  times_pregnant  0.0000
            Column   Value
0   times_pregnant  1.0000
9   times_pregnant  1.0000
10  times_pregnant  1.0000
19  times_pregnant  1.0000
20  times_pregnant  5.0000


In [39]:
# Needs to drop for both: 9, 19, 29, ...
splitted_odds_without_duplication = splitted_odds.drop(list(range(9, len(splitted_odds), 10)))
splitted_odds_without_duplication = splitted_odds_without_duplication.reset_index(drop=True)
splitted_odds_without_duplication.iloc[8:11]

,Column,Value
8,diabetes,1.0000
9,times_pregnant,8.0000
10,plasma_glucose_concentration,183.0000


In [40]:
splitted_evens_without_duplication = splitted_evens.drop(list(range(9, len(splitted_evens), 10)))
splitted_evens_without_duplication = splitted_evens_without_duplication.reset_index(drop=True)
splitted_evens_without_duplication.iloc[8:11]

,Column,Value
8,diabetes,0.0000
9,times_pregnant,1.0000
10,plasma_glucose_concentration,89.0000


In [42]:
def make_ready_for_pivot(df):
    # Add row for grouping
    ready_to_pivot = df.copy()
    ready_to_pivot['Row'] = df.index // len(changed_col_names)
    ready_to_pivot['Value'] = pd.to_numeric(ready_to_pivot['Value'], errors='coerce')
    print(f"Have none: {ready_to_pivot.isna().any().any()}")
    print(f"Column types: {ready_to_pivot.dtypes}")
    return ready_to_pivot

ready_odds = make_ready_for_pivot(splitted_odds_without_duplication)
ready_odds.head(10)

Have none: False
Column types: Column     object
Value     float64
Row         int64
dtype: object


,Column,Value,Row
0,times_pregnant,6.000,0
1,plasma_glucose_concentration,148.000,0
2,diastolic_blood_pressure,72.000,0
3,triceps_thickness,35.000,0
4,2_hour_serum_insulin,0.000,0
5,BMI,33.600,0
6,diabetes_pedigreen,0.627,0
7,age,50.000,0
8,diabetes,1.000,0
9,times_pregnant,8.000,1


In [43]:
ready_evens = make_ready_for_pivot(splitted_evens_without_duplication)
ready_evens.head(10)

Have none: False
Column types: Column     object
Value     float64
Row         int64
dtype: object


,Column,Value,Row
0,times_pregnant,1.000,0
1,plasma_glucose_concentration,85.000,0
2,diastolic_blood_pressure,66.000,0
3,triceps_thickness,29.000,0
4,2_hour_serum_insulin,0.000,0
5,BMI,26.600,0
6,diabetes_pedigreen,0.351,0
7,age,31.000,0
8,diabetes,0.000,0
9,times_pregnant,1.000,1


None of them has any NaN values, Value colum is float, index starts from times_pregnant

In [44]:
pivoted_odds = ready_odds.pivot(index = 'Row', columns = 'Column', values = 'Value')
print(pivoted_odds.isna().any().any())
pivoted_odds.head()

False


Column,2_hour_serum_insulin,BMI,age,diabetes,diabetes_pedigreen,diastolic_blood_pressure,plasma_glucose_concentration,times_pregnant,triceps_thickness
Row,,,,,,,,,
0,0.0,33.6,50.0,1.0,0.627,72.0,148.0,6.0,35.0
1,0.0,23.3,32.0,1.0,0.672,64.0,183.0,8.0,0.0
2,168.0,43.1,33.0,1.0,2.288,40.0,137.0,0.0,35.0
3,88.0,31.0,26.0,1.0,0.248,50.0,78.0,3.0,32.0
4,543.0,30.5,53.0,1.0,0.158,70.0,197.0,2.0,45.0


In [45]:
pivoted_evens = ready_evens.pivot(index = 'Row', columns = 'Column', values = 'Value')
print(pivoted_evens.isna().any().any())
pivoted_evens.head()

False


Column,2_hour_serum_insulin,BMI,age,diabetes,diabetes_pedigreen,diastolic_blood_pressure,plasma_glucose_concentration,times_pregnant,triceps_thickness
Row,,,,,,,,,
0,0.0,26.6,31.0,0.0,0.351,66.0,85.0,1.0,29.0
1,94.0,28.1,21.0,0.0,0.167,66.0,89.0,1.0,23.0
2,0.0,25.6,30.0,0.0,0.201,74.0,116.0,5.0,0.0
3,0.0,35.3,29.0,0.0,0.134,0.0,115.0,10.0,0.0
4,0.0,37.6,30.0,0.0,0.191,92.0,110.0,4.0,0.0


In [46]:
# Concat them!
max_len = max(len(pivoted_evens), len(pivoted_odds))
pivoted_messy = pd.concat(
                    [pivoted_odds[:max_len], 
                     pivoted_evens[:max_len]]
                  ).sort_index(kind='merge').reset_index(drop=True)
pivoted_messy

Column,2_hour_serum_insulin,BMI,age,diabetes,diabetes_pedigreen,diastolic_blood_pressure,plasma_glucose_concentration,times_pregnant,triceps_thickness
0,0.0,33.6,50.0,1.0,0.627,72.0,148.0,6.0,35.0
1,0.0,26.6,31.0,0.0,0.351,66.0,85.0,1.0,29.0
2,0.0,23.3,32.0,1.0,0.672,64.0,183.0,8.0,0.0
3,94.0,28.1,21.0,0.0,0.167,66.0,89.0,1.0,23.0
4,168.0,43.1,33.0,1.0,2.288,40.0,137.0,0.0,35.0
...,...,...,...,...,...,...,...,...,...
763,0.0,22.5,33.0,0.0,0.142,62.0,89.0,9.0,0.0
764,180.0,32.9,63.0,0.0,0.171,76.0,101.0,10.0,48.0
765,0.0,36.8,27.0,0.0,0.340,70.0,122.0,2.0,27.0
766,112.0,26.2,30.0,0.0,0.245,72.0,121.0,5.0,23.0


In [47]:
# Rename to original
pivoted_messy.rename(columns={"2_hour_serum_insulin": "2-hour_serum_insulin"}, inplace=True)
pivoted_messy.head()

Column,2-hour_serum_insulin,BMI,age,diabetes,diabetes_pedigreen,diastolic_blood_pressure,plasma_glucose_concentration,times_pregnant,triceps_thickness
0,0.0,33.6,50.0,1.0,0.627,72.0,148.0,6.0,35.0
1,0.0,26.6,31.0,0.0,0.351,66.0,85.0,1.0,29.0
2,0.0,23.3,32.0,1.0,0.672,64.0,183.0,8.0,0.0
3,94.0,28.1,21.0,0.0,0.167,66.0,89.0,1.0,23.0
4,168.0,43.1,33.0,1.0,2.288,40.0,137.0,0.0,35.0


In [48]:
column_names = non_messy_data_df.columns.to_list()
# Reorder columns
pivoted_messy = pivoted_messy[column_names]
pivoted_messy[column_names[:5]] = pivoted_messy[column_names[:5]].astype(int)
pivoted_messy[column_names[7:9]] = pivoted_messy[column_names[7:9]].astype(int)
pivoted_messy.dtypes
# Add col and row name
pivoted_messy.columns.name = non_messy_data_df.columns.name
pivoted_messy.index.name = non_messy_data_df.index.name

In [49]:
pivoted_messy.head()

,times_pregnant,plasma_glucose_concentration,diastolic_blood_pressure,triceps_thickness,2-hour_serum_insulin,BMI,diabetes_pedigreen,age,diabetes
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [50]:
# Compare columns
columns_diff = set(pivoted_messy.columns).symmetric_difference(set(non_messy_data_df.columns))
print("Columns not matching:", columns_diff)

# Compare indexes
index_diff = set(pivoted_messy.index).symmetric_difference(set(non_messy_data_df.index))
print("Indexes not matching:", index_diff)

Columns not matching: set()
Indexes not matching: set()


In [51]:
pivoted_messy.head()

,times_pregnant,plasma_glucose_concentration,diastolic_blood_pressure,triceps_thickness,2-hour_serum_insulin,BMI,diabetes_pedigreen,age,diabetes
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [52]:
non_messy_data_df.head()

,times_pregnant,plasma_glucose_concentration,diastolic_blood_pressure,triceps_thickness,2-hour_serum_insulin,BMI,diabetes_pedigreen,age,diabetes
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [53]:
# It's not equal everywhere, so probably the rows should have been merged in a different order
# I'll check back with the unit testing
if not non_messy_data_df.equals(pivoted_messy):
    print((non_messy_data_df != pivoted_messy).stack()[lambda x: x].index)

MultiIndex([(  9,               'times_pregnant'),
            (  9, 'plasma_glucose_concentration'),
            (  9,     'diastolic_blood_pressure'),
            (  9,                          'BMI'),
            (  9,           'diabetes_pedigreen'),
            (  9,                          'age'),
            (  9,                     'diabetes'),
            ( 10,               'times_pregnant'),
            ( 10, 'plasma_glucose_concentration'),
            ( 10,     'diastolic_blood_pressure'),
            ...
            (765,                          'age'),
            (766,               'times_pregnant'),
            (766, 'plasma_glucose_concentration'),
            (766,     'diastolic_blood_pressure'),
            (766,            'triceps_thickness'),
            (766,         '2-hour_serum_insulin'),
            (766,                          'BMI'),
            (766,           'diabetes_pedigreen'),
            (766,                          'age'),
            (76

## Part 2: Unit Testing

Below, are a series of simple arithmetic functions. Define class of test-cases for these functions that will adequately assure you they are working properly.

Hopefully you implemented Part 1 using a pipeline of functions. Here, you should design and implement unit tests for each function. Be sure to test edge-cases with values not neccesarily observed in the dataset. You may have to refer to the Python unittest package documentations: https://docs.python.org/3/library/unittest.html

In [20]:
#Should you want to delete a test case from within Jupyter notebook,
#you can run the following code to remove the class from the set of
#global variables: 

In [21]:
def incr(x):
    return x + 1

def decr(x):
    return x - 1

def add(x,y):
    return x + y

def subt(x):
    return x - 1

def mult(x,y):
    return x * y

def divi(x):
    return x / y

In [30]:
# del IncrTestCast
import unittest

class myTest(unittest.TestCase):
    def test_increase_exists(self):
        self.assertIsNotNone(incr)

    def test_increase_numeric(self):
        ret = incr(3)
        self.assertIsInstance(ret, (int, float))

    def test_increase_res(self):
        ret = incr(3)
        self.assertEqual(ret, 4)

    def test_dec_exists(self):
        self.assertIsNotNone(decr)

    def test_dec_numeric(self):
        ret = decr(3)
        self.assertIsInstance(ret, (int, float))

    def test_dec_res(self):
        ret = decr(3)
        self.assertEqual(ret, 2)

# This strange Python simulates running your code as if it were executed
# from the command-line, instead of within a Notebook. All it does is 
# call the automatically generated main() function (which is usually
# wrapped in Jupyter) with an explicit argument array with one value.
if __name__ == '__main__':
    unittest.main(argv=[''], exit=False,verbosity=2)

test_dec_exists (__main__.myTest.test_dec_exists) ... ok
test_dec_numeric (__main__.myTest.test_dec_numeric) ... ok
test_dec_res (__main__.myTest.test_dec_res) ... ok
test_increase_exists (__main__.myTest.test_increase_exists) ... ok
test_increase_numeric (__main__.myTest.test_increase_numeric) ... ok
test_increase_res (__main__.myTest.test_increase_res) ... ok

----------------------------------------------------------------------
Ran 6 tests in 0.011s

OK
